# BLOQUE 01 – EQUIPAMIENTOS 
## Objetivo
Construir capa consolidada EQUIPAMIENTOS normalizada por unidad espacial.
## Inputs
- equipamientos.shp
- manzana.shp

## Procesos
- Limpieza
- Homologación de CRS
- Join espacial
- Normalización

## Output

1_D3C2_output_1_Equipamientos.gpkg
- 1_D3C2V1_equipamientos.shp
- 1_D3C2V1_Mzn_equipamientos.shp

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon
from dbfread import DBF
import jenkspy
import json
from jenkspy import jenks_breaks
import time

In [18]:
# Rutas
eq_ruta       = r"C:\Users\Alejandro\Desktop\DAL_LAB\data_raw\Equipamientos\14sip.shp"
manzanas_ruta = r"C:\Users\Alejandro\Desktop\DAL_LAB\data_raw\Manzanas\14m.shp"

Diccionario_comentado = r"C:\Users\Alejandro\Desktop\DAL_LAB\data_raw\Equipamientos\Diccionario_pond_2020.xlsx"
Diccionario_nuevo = r"C:\Users\Alejandro\Desktop\DAL_LAB\data_raw\Equipamientos\Diccionario_2026.xlsx"
Diccionario_actualizado = r"C:\Users\Alejandro\Desktop\DAL_LAB\data_raw\Equipamientos\Diccionario_pond_2026.xlsx"

ruta_salida   = r"C:\Users\Alejandro\Desktop\DAL_LAB\outputs\etapa_1\1_D3C2_output_1_Equipamientos.gpkg"

In [3]:
# Cargar los GeoDataFrames
gdf_eq = gpd.read_file(eq_ruta)
gdf_manzanas = gpd.read_file(manzanas_ruta)

In [4]:
gdf_eq = gdf_eq.to_crs(gdf_manzanas.crs)

In [8]:
gdf_eq.columns

Index(['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'CVE_LOC', 'CVE_AGEB', 'CVE_MZA',
       'CONDICION', 'GEOGRAFICO', 'NOMSERV', 'TIPO', 'CVE_SERV', 'AMBITO',
       'geometry'],
      dtype='object')

In [10]:
gdf_eq = gdf_eq.drop(columns=['CVE_SERV', 'AMBITO'])

## Actualización del diccionario

In [13]:
# carga de diccionario viejo
diccionario = pd.read_excel(Diccionario_comentado)

#crear nuevo dic

# Agrupar por 'GEOGRÁFICO' y obtener los valores únicos de 'TIPO' para cada uno
diccionario_geografico_tipo = {}

# Recorrer cada valor único de GEOGRAFICO
for geografico in gdf_eq['GEOGRAFICO'].unique():
    # Filtrar los datos correspondientes al valor de 'GEOGRAFICO' actual
    subset = gdf_eq[gdf_eq['GEOGRAFICO'] == geografico]
    
    # Obtener los valores únicos de 'TIPO' asociados a este 'GEOGRAFICO'
    tipos = subset['TIPO'].unique()
    
    # Añadir al diccionario, si no tiene subniveles, ponemos una lista vacía
    diccionario_geografico_tipo[geografico] = list(tipos) if len(tipos) > 0 else []

# Mostrar el diccionario creado
for geo, tipos in diccionario_geografico_tipo.items():
    print(f"{geo}: {tipos}")

# Convertir el diccionario en un DataFrame
data = []

# Recorrer el diccionario y estructurarlo en filas
for geografico, tipos in diccionario_geografico_tipo.items():
    if tipos:  # Si hay subniveles en 'TIPO'
        for tipo in tipos:
            data.append([geografico, tipo])
    else:
        data.append([geografico, None])  # Si no hay subniveles, ponemos 'None' en 'TIPO'

Plaza: ['No Aplica']
Escuela: ['Media Superior', 'Secundaria', 'Primaria', 'Preescolar', 'Mixto', 'Superior']
Templo: ['No Aplica']
Centro de Asistencia Médica: ['Centro de Salud', 'Hospital', 'Centro de Rehabilitación']
Instalación Deportiva o Recreativa: ['Cancha', 'Unidad Deportiva', 'Lienzo Charro', 'Parque', 'Plaza de Toros', 'Balneario', 'Jardín', 'Campo de Golf', 'Estadio', 'Áreas Verdes', 'Velódromo', 'Acuario', 'Zoológico', 'Campo de Tiro', 'Hipódromo']
Pozo: ['Agua']
Tanque de Agua: ['Tanque Elevado', 'Caja de Agua']
Cementerio: ['No Aplica']
Instalación Gubernamental: ['Palacio Municipal', 'Palacio de Gobierno']
Mercado: ['No Aplica']
Instalación de Servicios: ['Centro de Asistencia Social', 'Central de Bomberos', 'Central de Policía']
Instalación Diversa: ['Silo', 'Edificación Cultural', 'Museo', 'Centro de Espectáculos', 'Reclusorio', 'Monumento u Obelisco', 'Centro de Investigación', 'Planta de Tratamiento de Agua', 'Teatro', 'Estación de Gas']
Instalación de Comunicación

In [14]:
# Crear DataFrame
df_diccionario = pd.DataFrame(data, columns=['GEOGRAFICO', 'TIPO'])

In [15]:
# Unir el diccionario nuevo con el viejo (asegurarse de que haya un campo en común 'GEOGRAFICO' y 'TIPO')
df_diccionario = df_diccionario.merge(diccionario, on=['GEOGRAFICO', 'TIPO'], how='left')

In [20]:
# Exportar a archivo Excel
df_diccionario.to_excel(Diccionario_nuevo, index=False)

print(f"Diccionario exportado exitosamente a {Diccionario_nuevo}")

Diccionario exportado exitosamente a C:\Users\Alejandro\Desktop\DAL_LAB\data_raw\Equipamientos\Diccionario_2026.xlsx


## Unir claves a equipamientos

In [11]:
diccionario = pd.read_excel(Diccionario_actualizado)

# Unir el diccionario con el geodataframe gdf_2020 (asegurarse de que haya un campo en común, por ejemplo 'GEOGRAFICO' y 'TIPO')
gdf_eq = gdf_eq.merge(diccionario, on=['GEOGRAFICO', 'TIPO'], how='left')

In [12]:
gdf_eq.columns

Index(['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'CVE_LOC', 'CVE_AGEB', 'CVE_MZA',
       'CONDICION', 'GEOGRAFICO', 'NOMSERV', 'TIPO', 'geometry', '#',
       'CATEGORIA', 'RADIO', 'IMP', 'IMP_B', 'IMP_C'],
      dtype='object')

In [ ]:
gdf_eq = gdf_eq.drop(columns=['IMP', 'IMP_B'])

In [ ]:
# Filtrar por categorías y exportar a archivos individuales

# Crear una lista de categorías
categorias = ['Educación', 'Salud', 'Recreación', 'Servicios', 'Comercio', 'Cultura', 'Transporte']

# Crear y exportar un GeoDataFrame por cada categoría a una capa dentro del geopackage
for categoria in categorias:
    # Filtrar los equipamientos de la categoría actual
    gdf_categoria = gdf_2020[gdf_2020['CATEGORIA'] == categoria]
    
    # Exportar la categoría como una nueva capa en el geopackage
    gdf_categoria.to_file(equipamientos_gpkg, layer=categoria, driver="GPKG")
    
    print(f"Exportado: {categoria} a la capa {categoria} en el geopackage")

## Tratado Denue

In [13]:
gdf_denue.columns

Index(['id', 'nom_estab', 'codigo_act', 'nombre_act', 'per_ocu', 'cve_ent',
       'cve_mun', 'cve_loc', 'ageb', 'manzana', 'fecha_alta', 'geometry',
       'Num_Emp'],
      dtype='object')

In [12]:
#Borrar columnas DENUE
gdf_denue = gdf_denue.drop(columns=['clee','raz_social', 
        'tipo_vial', 'nom_vial', 'tipo_v_e_1', 'nom_v_e_1',
       'tipo_v_e_2', 'nom_v_e_2', 'tipo_v_e_3', 'nom_v_e_3', 'numero_ext',
       'letra_ext', 'edificio', 'edificio_e', 'numero_int', 'letra_int',
       'tipo_asent', 'nomb_asent', 'tipoCenCom', 'nom_CenCom', 'num_local',
       'cod_postal','entidad',  'municipio',
       'localidad',  'telefono', 'correoelec', 'www',
       'tipoUniEco', 'latitud', 'longitud'])

In [4]:
#Campo empleados

# Definir la función de conversión
def convert_per_ocu_to_value(per_ocu):
    mapping = {
        '0 a 5 personas': 3,
        '6 a 10 personas': 8,
        '11 a 30 personas': 20,
        '31 a 50 personas': 40,
        '51 a 100 personas': 75,
        '101 a 250 personas': 175,
        '251 y más personas': 275
    }
    return mapping.get(per_ocu, None)

# Aplicar la función para crear la nueva columna
gdf_denue['Num_Emp'] = gdf_denue['per_ocu'].apply(convert_per_ocu_to_value)

In [29]:
# Guardar el DENUE con los nuevos campos
gdf_denue.to_file(ruta_salida, layer='1_D3C1V1_Unidades_económicas', driver='GPKG')

## Tratado Manzanas

In [27]:
# Crear un buffer de 10 metros alrededor de las manzanas
gdf_manzanas_buffer = gdf_manzanas.copy()
gdf_manzanas_buffer['geometry'] = gdf_manzanas_buffer.geometry.buffer(10)

# Realizar la unión espacial entre las manzanas y las unidades económicas
gdf_union = gpd.sjoin(gdf_manzanas_buffer, gdf_denue, how="left", predicate="contains")

# Contar el número de unidades económicas dentro de cada manzana
numero_unidades_por_manzana = gdf_union.groupby('CVEGEO').size().reset_index(name='UE')

# Unir los resultados al GeoDataFrame de las manzanas
gdf_manzanas_final = gdf_manzanas.merge(numero_unidades_por_manzana, on='CVEGEO', how='left')

# Mostrar el resultado
gdf_manzanas_final

,CVEGEO,CVE_ENT,CVE_MUN,CVE_LOC,CVE_AGEB,CVE_MZA,AMBITO,TIPOMZA,geometry,UE
0,1400100010226003,14,001,0001,0226,003,Urbana,Típica,"POLYGON ((2406377.75 974961.594, 2406376.127 9...",4
1,1400100010211015,14,001,0001,0211,015,Urbana,Típica,"POLYGON ((2406681.686 975531.995, 2406674.665 ...",1
2,1400100010211011,14,001,0001,0211,011,Urbana,Típica,"POLYGON ((2406061.676 975289.259, 2406069.062 ...",13
3,1400100010207020,14,001,0001,0207,020,Urbana,Típica,"POLYGON ((2405836.806 974518.055, 2405844.876 ...",8
4,1400100010194015,14,001,0001,0194,015,Urbana,Típica,"POLYGON ((2405774.76 975251.892, 2405774.328 9...",5
...,...,...,...,...,...,...,...,...,...,...
140999,141250052012A040,14,125,0052,012A,040,Rural,Típica,"POLYGON ((2450269.737 963843.34, 2450268.438 9...",1
141000,141250052012A041,14,125,0052,012A,041,Rural,Típica,"POLYGON ((2450228.916 963905.119, 2450273.154 ...",1
141001,141250052012A042,14,125,0052,012A,042,Rural,Típica,"POLYGON ((2450273.461 963909.843, 2450228.419 ...",1
141002,1412500640115025,14,125,0064,0115,025,Rural,Típica,"POLYGON ((2445003.343 977547.067, 2444993.892 ...",1


In [28]:
# Guardar el GeoPackage con los nuevos campos
gdf_manzanas_final.to_file(ruta_salida, layer='1_D3C1V1_Mzn_Unidades_Economicas', driver='GPKG')